# Study report: stimulus-sampling

The study asks if the random stimulus samples the latency evenly over the
kernel tick, and how many samples, stimulus seeds and load seeds a value
from the p50 to the max needs.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-measurement-validity@nucleo_g474re/st-stimulus-sampling"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "probatrix" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## Tick phase coverage

The share of the edges in each of the 64 phase bins of the tick, against
the uniform share (1 = even). `random` is the stimulus of all
experiments; `fixed-scan` moves the phase by 1 cycle per edge.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
k = 0
for exp in ("irq-phase-1m", "wake-phase-1m"):
    for run, tree, point, rep in inst.points(exp):
        for metric, ph in rep["phases"].items():
            tot = sum(b[1] for b in ph["bins"])
            fig.add_trace(go.Scatter(x=[b[0] for b in ph["bins"]],
                                     y=[b[1] * len(ph["bins"]) / tot for b in ph["bins"]], mode="lines+markers",
                                     name=f"{exp} {point} {metric}", line={"color": nbfig.color(k), "width": 2},
                                     marker={"size": 6}))
            k += 1
nbfig.style(fig, "Edges per phase bin of the tick", "phase bin (of 64)", "share / uniform share",
            hovermode="x unified").show()

## CCDF by stimulus seed (wake-stim-seed-1m)

Eight stimulus seeds at the same point. The dashed line is the point
`baseline-t64` of `wake-kconfig-1m` (study kconfig-factors, imported):
the same configuration, another run.

In [ ]:
series = nbfig.hists_of(inst, "wake-stim-seed-1m")
if nbfig.built(inst, "wake-kconfig-1m"):
    series["wake-kconfig-1m baseline-t64"] = next(rep["hists"] for _, _, p, rep in inst.points("wake-kconfig-1m")
                                                  if p == "baseline-t64")
nbfig.ccdf(series, "wake-stim-seed-1m: CCDF by stimulus seed", ["irq", "isr2thr", "irq2thr"],
           dashed={"wake-kconfig-1m baseline-t64"}).show()

In [ ]:
df = inst.rows("wake-stim-seed-1m")
df[df.metric == "irq"][["point", "p50", "p99", "p99.9", "p99.99", "max", "mean"]]

## CCDF by load seed (wake-load-seed-tail)

The 4M run and three load seeds of 1M samples. The CCDF is a share, so
runs of different length are comparable.

In [ ]:
lst = inst.points("wake-load-seed-tail")
nbfig.ccdf(nbfig.hists_of(inst, "wake-load-seed-tail"), "wake-load-seed-tail: CCDF by load seed",
           ["irq", "isr2thr", "irq2thr"]).show()

## Tail count against the ticks with many expiries

H3 a..c predicted that the count of samples >= 4288 cycles (per 1M
samples) grows with the ticks with >= 3 load timer expiries (from
`loadmodel.py`). The dashed line is that prediction, through the mean of
the 8 stimulus seeds.

In [ ]:
import loadmodel  # noqa: E402

THRESHOLD, K = 4288, 3


def tail_row(label, rep):
    """Ticks with >= K expiries (model) and samples >= THRESHOLD, both per 1M samples."""
    params = loadmodel.log_params({k: str(v) for k, v in rep["meta"].items()})
    ticks = int(params["seconds"] * params["ticks_per_sec"]) + 1
    tset = loadmodel.timers(params["seed"], params["timers"], params["min_ms"], params["max_ms"],
                            params["ticks_per_sec"])
    s = loadmodel.summary(loadmodel.expiries_per_tick(tset, ticks), 1)
    h = rep["hists"]["irq"]
    count = sum(c for v, c in h["bins"] if v >= THRESHOLD) + sum(c for v, c, _ in h["coarse"] if v >= THRESHOLD)
    per_m = 1e6 / h["n"]
    return {"point": label, "load_seed": f"0x{params['seed']:08x}",
            "ticks": loadmodel.at_least(s["counts"], K) * per_m, "samples": count * per_m}


df = pd.DataFrame([tail_row(point, rep) for _, _, point, rep in lst])
seeds = pd.DataFrame([tail_row(point, rep) for _, _, point, rep in inst.points("wake-stim-seed-1m")])
x0, y0 = seeds.ticks.mean(), seeds.samples.mean()
fig = go.Figure(go.Scatter(x=df.ticks, y=df.samples, mode="markers+text", text=df.point,
                           textposition="top center", marker={"size": 10, "color": nbfig.color(0)},
                           name="wake-load-seed-tail"))
fig.add_trace(go.Scatter(x=[x0], y=[y0], mode="markers", marker={"symbol": "diamond", "size": 10,
                                                                  "color": nbfig.color(1)},
                         error_y={"type": "data", "symmetric": False, "array": [seeds.samples.max() - y0],
                                  "arrayminus": [y0 - seeds.samples.min()]},
                         name="wake-stim-seed-1m: mean and range of 8 seeds"))
xs = [0, df.ticks.max() * 1.1]
fig.add_trace(go.Scatter(x=xs, y=[x * y0 / x0 for x in xs], mode="lines",
                         line={"dash": "dash", "color": nbfig.color(2)}, name="proportional to the ticks (H3)"))
nbfig.style(fig, "Tail count against the ticks with many expiries",
              f"ticks with >= {K} expiries per 1M samples (loadmodel.py)",
              f"samples >= {THRESHOLD} cycles per 1M").show()
df